In [2]:
import pandas as pd
import numpy as np
import os
from typing import Tuple
import ast

def load_protocol_data(
    protocol: str,
    loan_token: str,
    *,
    morpho_market: str = "CBBTC",
    galaxy_type: str = "no-class-a"
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Loads user-level and market-level protocol data.

    Reads 'users_df.parquet' and 'market_df.parquet' from the input folder.

    Returns:
        A tuple of (users_df, market_df):
        - users_df: DataFrame with per-user position data.
        - market_df: DataFrame with market-level metrics (this is important only to understand which collateral are suitable for the calibration process, since we upload prices through an API).
    """
    _path = lambda filename: os.path.join("inputs", filename)
    protocol = protocol.lower()
    if protocol == "morpho":
        loan_token = f"{morpho_market}-{loan_token}".lower()
        users_df = pd.read_parquet(_path(f"users_{protocol}_{loan_token}.parquet"))
        market_df = pd.read_parquet(_path(f"market_{protocol}_{loan_token}.parquet"))
    
    elif protocol == "galaxy":
        type = galaxy_type.lower()
        users_df = pd.read_parquet(_path(f"users_{protocol}_{type}.parquet"))
        market_df = pd.read_parquet(_path(f"market_{protocol}.parquet"))
    
    
    elif protocol == "anchorage":
        users_df = pd.read_parquet(_path(f"users_{protocol}.parquet"))
        market_df = pd.read_parquet(_path(f"market_{protocol}.parquet"))
    
    else:
        users_df = pd.read_parquet(_path(f"users_{protocol}_{loan_token}.parquet"))
        market_df = pd.read_parquet(_path(f"market_{protocol}_{loan_token}.parquet"))
    
    
    return users_df, market_df

PROTOCOL = "MORPHO"
MORPHO_MARKET = "WETH"
LOAN_TOKEN = "USDC"

# Example usage
users_df, market_df = load_protocol_data(
    PROTOCOL, 
    LOAN_TOKEN, 
    morpho_market=MORPHO_MARKET,
    galaxy_type="no-class-a"
)


In [3]:
PROTOCOL_LIST = ["MORPHO", "GALAXY", "ANCHORAGE", "SYRUP", "SPARKLEND"]
BORROW_LIST = ["USDC", "DAI", "USDT", "USDS", "SUSDS", "SDAI"]

def load_all_protocol_data(
    protocol_list: list[str] = PROTOCOL_LIST,
    collateral_list: list[str] = BORROW_LIST,
    morpho_market: str = "CBBTC",
    galaxy_type: str = "no-class-a"
) -> dict[Tuple[str, str], Tuple[pd.DataFrame, pd.DataFrame]]:
    """
    Loads user-level and market-level protocol data for all combinations of protocols and collaterals.

    Returns:
        A dictionary mapping (protocol, collateral) to (users_df, market_df).
    """
    data = {}
    for protocol in protocol_list:
        for collateral in collateral_list:
            try:
                users_df, market_df = load_protocol_data(
                    protocol,
                    collateral,
                    morpho_market=morpho_market,
                    galaxy_type=galaxy_type
                )
                data[(protocol, collateral)] = (users_df, market_df)
            except FileNotFoundError:
                print(f"Data for {protocol} with collateral {collateral} not found. Skipping.")
    return data

# Usage
all_data = load_all_protocol_data()

unique_tokens = set()

for (users_df, _) in all_data.values():
    for val in users_df['collateral_token_symbol']:

        # Case 1: numpy array
        if isinstance(val, np.ndarray):
            unique_tokens.update(val.tolist())

        # Case 2: list
        elif isinstance(val, list):
            unique_tokens.update(val)

        # Case 3: string that looks like a list
        elif isinstance(val, str) and val.startswith("["):
            try:
                parsed = ast.literal_eval(val)
                if isinstance(parsed, (list, tuple, np.ndarray)):
                    unique_tokens.update(list(parsed))
                else:
                    unique_tokens.add(parsed)
            except:
                pass

        # Case 4: plain scalar
        else:
            unique_tokens.add(val)

collateral_tokens = [token for token in unique_tokens if token not in BORROW_LIST]
print(collateral_tokens)

Data for MORPHO with collateral DAI not found. Skipping.
Data for MORPHO with collateral USDT not found. Skipping.
Data for MORPHO with collateral USDS not found. Skipping.
Data for MORPHO with collateral SUSDS not found. Skipping.
Data for MORPHO with collateral SDAI not found. Skipping.
Data for SYRUP with collateral DAI not found. Skipping.
Data for SYRUP with collateral USDS not found. Skipping.
Data for SYRUP with collateral SUSDS not found. Skipping.
Data for SYRUP with collateral SDAI not found. Skipping.
Data for SPARKLEND with collateral SUSDS not found. Skipping.
Data for SPARKLEND with collateral SDAI not found. Skipping.
['WBTC', 'LBTC', 'WETH', 'XRP', 'TBTC', 'WSTETH', 'HYPE', 'CBBTC', 'RETH', 'EZETH', 'BTC', 'WEETH', 'RSETH']


In [4]:
# LOAD ALL PRICES FROM YAHOO FINANCE
import yfinance as yf
from typing import Optional, List, Tuple

def load_data_yahoo(
    ticker: str,
    start_date: Optional[int] = None,
    end_date: Optional[int] = None,
    time_interval: str = "1d",
    period: Optional[str] = None
) -> pd.DataFrame:

    ticker_map = {
        "CBBTC": "CBBTC32994",
        "HYPE": "HYPE32196",
        "LBTC": "LBTC33652",
        "ARB": "ARB11841",
        "TBTC": "TBTC26133"
    }

    ticker = ticker_map.get(ticker, ticker)
    yahoo_ticker = ticker + "-USD"

    download_kwargs = {
        "interval": time_interval,
        "auto_adjust": False,
        "progress": False
    }

    # Choose either period OR start/end
    if period is not None:
        download_kwargs["period"] = period
    else:
        download_kwargs["start"] = start_date
        download_kwargs["end"] = end_date

    prices = yf.download(yahoo_ticker, **download_kwargs)

    prices.columns = prices.columns.get_level_values(0)

    freq_ = f"{time_interval}in" if "m" in time_interval else time_interval

    expected_index = pd.date_range(
        start=prices.index[0],
        end=prices.index[-1],
        freq=freq_
    )

    prices = prices.reindex(expected_index)
    prices = prices.interpolate().ffill().bfill().dropna()

    return prices, yahoo_ticker

collateral_tokens.extend(["ETH", "SOL", "JITOSOL"])
print(collateral_tokens)

all_prices = {}
for token in collateral_tokens:
    try:
        prices, yahoo_ticker = load_data_yahoo(token, period="max")
        all_prices[token] = (prices, yahoo_ticker)
    except Exception as e:
        print(f"Could not load price for {token}: {e}")

all_prices_df = pd.DataFrame({
    token: data[0]['Close'] for token, data in all_prices.items()
})
print(all_prices_df.head())
all_prices_df = all_prices_df.iloc[:-7]

# all_prices_df.to_parquet("prices_df.parquet")

['WBTC', 'LBTC', 'WETH', 'XRP', 'TBTC', 'WSTETH', 'HYPE', 'CBBTC', 'RETH', 'EZETH', 'BTC', 'WEETH', 'RSETH', 'ETH', 'SOL', 'JITOSOL']
            WBTC  LBTC  WETH  XRP  TBTC  WSTETH  HYPE  CBBTC  RETH  EZETH  \
2014-09-17   NaN   NaN   NaN  NaN   NaN     NaN   NaN    NaN   NaN    NaN   
2014-09-18   NaN   NaN   NaN  NaN   NaN     NaN   NaN    NaN   NaN    NaN   
2014-09-19   NaN   NaN   NaN  NaN   NaN     NaN   NaN    NaN   NaN    NaN   
2014-09-20   NaN   NaN   NaN  NaN   NaN     NaN   NaN    NaN   NaN    NaN   
2014-09-21   NaN   NaN   NaN  NaN   NaN     NaN   NaN    NaN   NaN    NaN   

                   BTC  WEETH  RSETH  ETH  SOL  JITOSOL  
2014-09-17  457.334015    NaN    NaN  NaN  NaN      NaN  
2014-09-18  424.440002    NaN    NaN  NaN  NaN      NaN  
2014-09-19  394.795990    NaN    NaN  NaN  NaN      NaN  
2014-09-20  408.903992    NaN    NaN  NaN  NaN      NaN  
2014-09-21  398.821014    NaN    NaN  NaN  NaN      NaN  


In [5]:
# Create a dataframe for each collateral's orderbook
import requests
from web3 import Web3
import json
from eth_abi.abi import decode
from decimal import Decimal, getcontext


def get_hyper_orderbook(
    token: str,
    *,
    nsig_figs: int = 2
) -> tuple[pd.DataFrame, pd.DataFrame]:
    
    def find_id(
        base_symbol: str, 
        quote_symbol: str
    ) -> any:
        response = requests.post("https://api.hyperliquid.xyz/info", json={"type": "spotMeta"})
        data = response.json()
        ind = None
        quote_index = None
        for token in data["tokens"]:
            if token["name"] == quote_symbol:
                quote_index = token["index"]
            if token["name"] == base_symbol:
                ind = token["index"]
        
        for i in data["universe"]:
            if i["tokens"] == [ind, quote_index]:
                return i["name"]
        return None

    symbol = find_id(token, "USDC")
    
    # print(f"{symbol} orderbook data")
    response = requests.post("https://api.hyperliquid.xyz/info", json={"type": "l2Book", "coin": symbol, "nSigFigs": nsig_figs, "mantissa": None})
    orderbook = response.json()

    sell_orderbook = pd.DataFrame(orderbook['levels'][0])
    buy_orderbook  = pd.DataFrame(orderbook['levels'][1])

    # ✅ Proper column renaming
    sell_orderbook = sell_orderbook.rename(columns={'px': 'price'})
    buy_orderbook  = buy_orderbook.rename(columns={'px': 'price'})

    # ✅ Force all relevant columns to numeric
    for df in [sell_orderbook, buy_orderbook]:
        df['price']  = pd.to_numeric(df['price'], errors='coerce')
        df['sz']     = pd.to_numeric(df['sz'], errors='coerce')

    # ✅ Compute liquidity (fully numeric now)
    sell_orderbook['liquidity'] = sell_orderbook['price'] * sell_orderbook['sz']
    buy_orderbook['liquidity']  = buy_orderbook['price'] * buy_orderbook['sz']

    return sell_orderbook, buy_orderbook


def get_univ3_orderbook(
    pool_address: str, 
    multicall_address: str
) -> tuple[pd.DataFrame, None]:

    getcontext().prec = 80

    def get_uniswap_v3_pool_state(
        w3, 
        pool_address, 
        multicall_address
    ) -> dict:
        pool_address = Web3.to_checksum_address(pool_address)
        multicall_address = Web3.to_checksum_address(multicall_address)

        pool = w3.eth.contract(address=pool_address, abi=UNISWAP_V3_POOL_ABI)
        multicall = w3.eth.contract(address=multicall_address, abi=MULTICALL3_ABI)

        calls = [
            (pool_address, pool.encode_abi("token0")),
            (pool_address, pool.encode_abi("token1")),
            (pool_address, pool.encode_abi("fee")),
            (pool_address, pool.encode_abi("tickSpacing")),
            (pool_address, pool.encode_abi("slot0")),
            (pool_address, pool.encode_abi("liquidity")),
        ]

        response = multicall.functions.aggregate(calls).call()
        return_data = response[1]

        slot0 = decode([i["internalType"] for i in pool.functions.slot0.abi["outputs"]], return_data[4])
        decoded = {
            "token0": decode(["address"], return_data[0]),
            "token1": decode(["address"], return_data[1]),
            "fee": decode(["uint24"], return_data[2]),
            "tick_spacing": decode(["int24"], return_data[3]),
            "slot0": slot0,
            "liquidity": decode(["uint128"], return_data[5]),
        }

        ERC20_ABI = [
            {"constant": True, "inputs": [], "name": "decimals", "outputs": [{"name": "", "type": "uint8"}], "type": "function"},
            {"constant": True, "inputs": [], "name": "symbol",   "outputs": [{"name": "", "type": "string"}], "type": "function"},
        ]

        token0 = decoded["token0"][0]
        token1 = decoded["token1"][0]

        token0 = Web3.to_checksum_address(token0)
        token1 = Web3.to_checksum_address(token1)

        token0_contract = W3.eth.contract(address=token0, abi=ERC20_ABI)
        token1_contract = W3.eth.contract(address=token1, abi=ERC20_ABI)

        dec0 = token0_contract.functions.decimals().call()
        dec1 = token1_contract.functions.decimals().call()

        return {
            "token0": decoded["token0"][0],
            "token1": decoded["token1"][0],
            "dec0": dec0,
            "dec1": dec1,
            "fee": decoded["fee"][0],
            "tick_spacing": decoded["tick_spacing"][0],
            "current_tick": decoded["slot0"][1],
            "sqrt_price_x96": decoded["slot0"][0],
            "liquidity": float(decoded["liquidity"][0]) / 1e8,
        }

    def get_uniswap_v3_pool_state_and_ticks(
            w3, 
            pool_address, 
            multicall_address, 
            tick_start, 
            tick_end
        ) -> dict:
        pool_address = Web3.to_checksum_address(pool_address)
        multicall_address = Web3.to_checksum_address(multicall_address)

        pool = w3.eth.contract(address=pool_address, abi=UNISWAP_V3_POOL_ABI)
        multicall = w3.eth.contract(address=multicall_address, abi=MULTICALL3_ABI)

        # Base state calls
        base_calls = [
            (pool_address, pool.encode_abi("token0")),
            (pool_address, pool.encode_abi("token1")),
            (pool_address, pool.encode_abi("fee")),
            (pool_address, pool.encode_abi("tickSpacing")),
            (pool_address, pool.encode_abi("slot0")),
            (pool_address, pool.encode_abi("liquidity")),
        ]

        # Temporary call to get tickSpacing for tick range logic
        tmp_response = multicall.functions.aggregate([base_calls[3]]).call()
        tick_spacing = decode(["int24"], tmp_response[1][0])[0]

        # Prepare tick calls
        tick_indices = list(range(tick_start, tick_end + 1, tick_spacing))
        tick_calls = [
            (pool_address, pool.encode_abi("ticks", args=[tick]))
            for tick in tick_indices
        ]

        # Combine base + tick calls
        all_calls = base_calls + tick_calls
        combined_response = multicall.functions.aggregate(all_calls).call()
        return_data = combined_response[1]

        # Decode base pool data
        token0 = decode(["address"], return_data[0])[0]
        token1 = decode(["address"], return_data[1])[0]
        fee = decode(["uint24"], return_data[2])[0]
        tick_spacing = decode(["int24"], return_data[3])[0]
        slot0_outputs = [o["internalType"] for o in pool.functions.slot0.abi["outputs"]]
        slot0 = decode(slot0_outputs, return_data[4])
        liquidity = decode(["uint128"], return_data[5])[0]

        # Decode tick data
        tick_outputs = [o["internalType"] for o in pool.functions.ticks(0).abi["outputs"]]
        tick_data = {}
        offset = len(base_calls)
        for i, raw in enumerate(return_data[offset:]):
            decoded_tick = decode(tick_outputs, raw)
            tick_data[tick_indices[i]] = {
                "liquidityNet": float(decoded_tick[1]) / 1e8,
            }

        return {
            "token0": token0,
            "token1": token1,
            "fee": fee,
            "tick_spacing": tick_spacing,
            "current_tick": slot0[1],
            "sqrt_price_x96": slot0[0],
            "liquidity": float(liquidity) / 1e8,
            "ticks": tick_data
        }

    def price_from_sqrt(
        sqrtPriceX96: int, 
        dec0: int, 
        dec1: int
    ) -> tuple[Decimal, Decimal]:
        """
        Returns:
        p1_per_0: how many token1 per 1 token0 (whole-token units)
        p0_per_1: how many token0 per 1 token1 (whole-token units)
        """
        from decimal import Decimal, getcontext
        getcontext().prec = 80
        s = Decimal(sqrtPriceX96)
        Q96 = 2**96
        p = (s / Decimal(Q96)) ** 2  # raw price (token1 per token0) in base units
        # Adjust to whole-token units (decimals):
        p1_per_0 = p * (Decimal(10) ** Decimal(dec0 - dec1))
        p0_per_1 = (Decimal(1) / p) * (Decimal(10) ** Decimal(dec1 - dec0))
        return p1_per_0, p0_per_1

    def get_liquidity_profile(
        state: dict, 
        price_0: Decimal, 
        tick_0: int
    ) -> pd.DataFrame:
        ticks = pd.DataFrame([
            {"tick": t, "liq_net": v["liquidityNet"]}
            for t, v in state["ticks"].items()
        ]).sort_values("tick")

        ticks["price"] = float(price_0) * (1.0001 ** (ticks["tick"] - tick_0))

        ticks["liquidity"] = ticks["liq_net"].cumsum()
        ticks["liq_density"] = ticks["liquidity"] / ticks["liquidity"].abs().sum()
        
        return ticks

    cwd = os.getcwd()

    # Load ABIs
    UNISWAP_V3_POOL_ABI = json.load(open(os.path.join(cwd, "inputs", "pool.json")))
    MULTICALL3_ABI = json.load(open(os.path.join(cwd, "inputs" ,"multicall3.json")))

    print(UNISWAP_V3_POOL_ABI)
    print(MULTICALL3_ABI)

    W3 = Web3(Web3.HTTPProvider("https://mainnet.base.org"))
    state = get_uniswap_v3_pool_state(
        W3, 
        pool_address, 
        multicall_address
    )

    state_tick = get_uniswap_v3_pool_state_and_ticks(
        W3, 
        pool_address, 
        multicall_address,
        tick_start = -75500,
        tick_end = -66000
    )

    _, usdc_per_btc = price_from_sqrt(
        state["sqrt_price_x96"], 
        state['dec0'],
        state['dec1']
    )

    ticks_df = get_liquidity_profile(
        state_tick, 
        usdc_per_btc,
        state['current_tick']
    )
    ticks_df = ticks_df.sort_values(by='price', ascending=False).reset_index(drop=True)

    return ticks_df, None


def get_cex_orderbook(
    base_asset: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    
    # API websites
    BITFINEX_BASE = "https://api-pub.bitfinex.com"
    CRYPTOCOM_BASE = "https://api.crypto.com/exchange/v1"
    HUOBI_BASE = "https://api.huobi.pro"
    KUCOIN_BASE = "https://api.kucoin.com"
    GATE_BASE = "https://api.gateio.ws"
    GATE_PREFIX = "/api/v4"
    BITGET_BASE = "https://api.bitget.com"
    COINBASE_BASE = "https://api.exchange.coinbase.com"
    BYBIT_BASE = "https://api.bybit.com"
    OKX_BASE = "https://www.okx.com"
    KRAKEN_BASE = "https://api.kraken.com"
    BINANCE_SPOT_BASE = "https://api.binance.com"

    # Special aliases for some exchanges
    KRAKEN_BASE_ALIAS = {
        "BTC": "XBT",
        "ETH": "XETH", 
        # add others if needed 
    }

    # add also USDC and USD in order to account to max liquidity 
    def build_symbols(
        token: str,
        token_usd: str = "USDT"
    ) -> tuple[dict, dict]:
        """
        Return a dict of exchange -> symbol/pair for the given base asset.
        We keep using USDT for most CEXs and USD where you already used it.
        """
        # Kraken special handling (e.g. BTC -> XBT)
        kraken_base = KRAKEN_BASE_ALIAS.get(token, token)

        symbols_token = {
            "binance": f"{token}{token_usd}",          # e.g. XRPUSDT, BTCUSDT
            "bybit": f"{token}{token_usd}",            # e.g. XRPUSDT
            "okx": f"{token}-{token_usd}",             # e.g. XRP-USDT
            "bitget": f"{token}{token_usd}",           # e.g. XRPUSDT
            "gate": f"{token}_{token_usd}",            # e.g. XRP_USDT
            "kucoin": f"{token}-{token_usd}",          # e.g. XRP-USDT
            "huobi": f"{token.lower()}{token_usd.lower()}",    # e.g. xrpusdt
            "cryptocom": f"{token}_{token_usd}"       # e.g. XRP_USDT
        }
        symbols_usd = {
            "coinbase": f"{token}-USD",        # e.g. XRP-USD, BTC-USD
            "kraken": f"{kraken_base}USD",     # e.g. XRPUSD, XBTUSD
            "bitfinex": f"t{token}USD"         # e.g. tXRPUSD, tBTCUSD
        }
        return symbols_token, symbols_usd

    def fetch_bitfinex_orderbook(
        symbol: str,
        precision: str = "P0",
        length: int = 100,
    ) -> tuple[list, list]:
        """
        Fetch L2 book from Bitfinex (price-aggregated).

        Endpoint: /v2/book/{symbol}/{precision}
        Query params:
        - len: number of price points ("1", "25", "100")

        For trading pairs, each entry is:
        [PRICE, COUNT, AMOUNT]
        AMOUNT > 0 -> bid
        AMOUNT < 0 -> ask
        """
        url = f"{BITFINEX_BASE}/v2/book/{symbol}/{precision}"
        params = {"len": length}
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        bids = []
        asks = []

        for entry in data:
            price, count, amount = entry
            price = float(price)
            amount = float(amount)
            if amount > 0:
                bids.append((price, amount))
            elif amount < 0:
                asks.append((price, abs(amount)))

        # Enforce proper sorting
        bids = sorted(bids, key=lambda x: x[0], reverse=True)  # high -> low
        asks = sorted(asks, key=lambda x: x[0])                # low -> high

        return bids, asks

    def fetch_cryptocom_orderbook(
        instrument_name: str, 
        depth: int = 50
    ) -> tuple[list, list]:
        """
        Fetch L2 book from Crypto.com Exchange.

        Endpoint: /public/get-book
        Params:
        - instrument_name: e.g. "XRP_USDT", "BTC_USDT", "BTCUSD-PERP"
        - depth: number of bids/asks to return (max 50)

        Response:
        {
            "code": 0,
            "result": {
            "depth": ...,
            "data": [{
                "bids": [[price, size, num_orders], ...],
                "asks": [[price, size, num_orders], ...],
                ...
            }],
            "instrument_name": ...
            }
        }
        """
        url = f"{CRYPTOCOM_BASE}/public/get-book"
        params = {
            "instrument_name": instrument_name,
            "depth": depth,
        }
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        if data.get("code") != 0:
            raise RuntimeError(f"Crypto.com error: {data.get('code')}")

        result = data["result"]
        entries = result["data"][0]  # single book entry for the instrument
        bids_raw = entries.get("bids", [])
        asks_raw = entries.get("asks", [])

        # bids/asks entries: [price, qty, num_orders] -> keep price & qty
        bids = [(float(p), float(q)) for p, q, *_ in bids_raw]
        asks = [(float(p), float(q)) for p, q, *_ in asks_raw]
        return bids, asks

    def fetch_huobi_orderbook(
        symbol: str, 
        depth: int = 150, 
        level: str = "step0"
    ) -> tuple[list, list]:
        """
        Fetch L2 book from Huobi spot.

        Endpoint: /market/depth
        Params:
        - symbol: e.g. "xrpusdt", "btcusdt" (all lowercase)
        - depth: 5, 10, 20, or (for step0) up to 150 (default 150 if omitted)
        - type: aggregation level, e.g. "step0" (no aggregation)

        Response:
        {
            "status": "ok",
            "tick": {
            "bids": [[price, size], ...],
            "asks": [[price, size], ...],
            ...
            }
        }
        """
        url = f"{HUOBI_BASE}/market/depth"
        params = {
            "symbol": symbol,
            "depth": depth,
            "type": level,
        }
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        if data.get("status") != "ok":
            raise RuntimeError(f"Huobi error: {data.get('status')}")

        tick = data.get("tick", {})
        bids_raw = tick.get("bids", [])
        asks_raw = tick.get("asks", [])

        # entries are [price, size], may be numbers or strings -> cast to float
        bids = [(float(p), float(q)) for p, q in bids_raw]
        asks = [(float(p), float(q)) for p, q in asks_raw]
        return bids, asks

    def fetch_kucoin_orderbook(
        symbol: str,
        trade_type: str = "SPOT",
        limit: str = "100",
    ) -> tuple[list, list]:
        """
        Fetch L2 book from KuCoin.

        Endpoint: /api/ua/v1/market/orderbook
        Query params:
        - tradeType: "SPOT" or "FUTURES"
        - symbol: e.g. "XRP-USDT", "BTC-USDT"
        - limit: "20", "50", "100", "FULL" (string)

        Response (data):
        {
            "bids": [["price", "size"], ...],  # high -> low
            "asks": [["price", "size"], ...],  # low -> high
            ...
        }
        """
        url = f"{KUCOIN_BASE}/api/ua/v1/market/orderbook"
        params = {
            "tradeType": trade_type,
            "symbol": symbol,
            "limit": limit,
        }
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        if data.get("code") != "200000":
            raise RuntimeError(f"KuCoin error: {data.get('code')}")

        book = data["data"]
        bids_raw = book.get("bids", [])
        asks_raw = book.get("asks", [])

        bids = [(float(p), float(q)) for p, q in bids_raw]
        asks = [(float(p), float(q)) for p, q in asks_raw]
        return bids, asks

    def fetch_gate_orderbook(
        currency_pair: str, 
        limit: int = 100, 
        interval: str = "0"
    ) -> tuple[list, list]:
        """
        Fetch L2 book from Gate.io spot.

        Endpoint: /spot/order_book
        Params:
        - currency_pair: e.g. "XRP_USDT", "BTC_USDT"
        - interval: "0" = no aggregation
        - limit: number of depth levels

        Response:
        {
            "asks": [["price", "size"], ...],
            "bids": [["price", "size"], ...],
            ...
        }
        """
        url = f"{GATE_BASE}{GATE_PREFIX}/spot/order_book"
        params = {
            "currency_pair": currency_pair,
            "interval": interval,
            "limit": limit,
        }
        headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
        }

        resp = requests.get(url, params=params, headers=headers, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        bids_raw = data.get("bids", [])
        asks_raw = data.get("asks", [])

        bids = [(float(p), float(q)) for p, q in bids_raw]
        asks = [(float(p), float(q)) for p, q in asks_raw]
        return bids, asks

    def fetch_bitget_orderbook(
        symbol: str, 
        depth_type: str = "step0", 
        limit: int = 150
    ) -> tuple[list, list]:
        """
        Fetch L2 book from Bitget spot.
        Endpoint: /api/v2/spot/market/orderbook
        Response:
            {
            "code": "00000",
            "msg": "success",
            "data": {
                "asks": [["price", "size"], ...],
                "bids": [["price", "size"], ...],
                "ts": "..."
            }
            }
        """
        url = f"{BITGET_BASE}/api/v2/spot/market/orderbook"
        params = {
            "symbol": symbol,
            "type": depth_type,  # e.g. step0
            "limit": limit,
        }
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        if data.get("code") != "00000":
            raise RuntimeError(f"Bitget error: {data.get('msg')}")

        book = data["data"]
        bids_raw = book.get("bids", [])
        asks_raw = book.get("asks", [])

        bids = [(float(p), float(q)) for p, q in bids_raw]
        asks = [(float(p), float(q)) for p, q in asks_raw]
        return bids, asks

    def fetch_coinbase_orderbook(
        product_id: str, 
        level: int = 2
    ) -> tuple[list, list]:
        """
        Fetch L2 book from Coinbase.
        Response has bids/asks: [price, size, num_orders]
        """
        url = f"{COINBASE_BASE}/products/{product_id}/book"
        params = {"level": level}
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        bids = [(float(p), float(q)) for p, q, _ in data["bids"]]
        asks = [(float(p), float(q)) for p, q, _ in data["asks"]]
        return bids, asks

    def fetch_bybit_orderbook(
        symbol: str, 
        limit: int = 1000
    ) -> tuple[list, list]:
        """
        Fetch L2 book from Bybit spot.
        Response result: { "b": bids, "a": asks },
        each as [price, size]
        """
        url = f"{BYBIT_BASE}/v5/market/orderbook"
        params = {"category": "spot", "symbol": symbol, "limit": limit}
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        if data.get("retCode") != 0:
            raise RuntimeError(f"Bybit error: {data.get('retMsg')}")

        result = data["result"]
        bids_raw = result.get("b", [])
        asks_raw = result.get("a", [])

        bids = [(float(p), float(q)) for p, q in bids_raw]
        asks = [(float(p), float(q)) for p, q in asks_raw]
        return bids, asks

    def fetch_okx_orderbook(
        inst_id: str, 
        depth: int = 400
    ) -> tuple[list, list]:
        """
        Fetch L2 book from OKX.
        Response data[0].bids/asks: [price, size, ..., ...]
        """
        url = f"{OKX_BASE}/api/v5/market/books"
        params = {"instId": inst_id, "sz": depth}
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        if data.get("code") != "0":
            raise RuntimeError(f"OKX error: {data.get('msg')}")

        book = data["data"][0]
        bids_raw = book["bids"]
        asks_raw = book["asks"]

        bids = [(float(p), float(q)) for p, q, *_ in bids_raw]
        asks = [(float(p), float(q)) for p, q, *_ in asks_raw]
        return bids, asks

    def fetch_kraken_orderbook(
        pair: str, 
        count: int = 100
    ) -> tuple[list, list]:
        """
        Fetch L2 book from Kraken.
        Response 'result' has a key like 'XXBTZUSD' -> { 'bids': [...], 'asks': [...] }
        Each level: [price, volume, timestamp]
        """
        url = f"{KRAKEN_BASE}/0/public/Depth"
        params = {"pair": pair, "count": count}
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()

        if data.get("error"):
            raise RuntimeError(f"Kraken error: {data['error']}")

        book = next(iter(data["result"].values()))
        bids_raw = book["bids"]
        asks_raw = book["asks"]

        bids = [(float(p), float(q)) for p, q, _ in bids_raw]
        asks = [(float(p), float(q)) for p, q, _ in asks_raw]
        return bids, asks

    def fetch_orderbook(
        symbol: str, 
        limit: int = 100000
    ) -> tuple[list, list]:
        """
        Fetch spot L2 order book snapshot from Binance.
        No API key required.
        """
        url = f"{BINANCE_SPOT_BASE}/api/v3/depth"
        params = {"symbol": symbol, "limit": limit}
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        # convert to float
        bids = [(float(p), float(q)) for p, q in data["bids"]]
        asks = [(float(p), float(q)) for p, q in data["asks"]]
        return bids, asks

    # TOTAL FUNCTION

    def aggregate_books(
        books: List[Tuple[str, List[Tuple[float, float]], List[Tuple[float, float]]]]
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        books: list of (venue_name, bids, asks),
            where bids/asks are [(price, qty), ...]

        Returns:
            synthetic_bids, synthetic_asks as [(price, qty), ...]
            sorted in trading order:
                bids: high -> low
                asks: low -> high
        """
        all_bids = []
        all_asks = []

        for _, bids, asks in books:
            all_bids.extend(bids)
            all_asks.extend(asks)

        syn_bids = (
            pd.DataFrame(all_bids, columns=["price", "sz"])
            .groupby("price", as_index=False)
            .sum()
            .sort_values("price", ascending=False)
        )

        syn_asks = (
            pd.DataFrame(all_asks, columns=["price", "sz"])
            .groupby("price", as_index=False)
            .sum()
            .sort_values("price", ascending=True)
        )

        syn_bids['liquidity'] = syn_bids['price'] * syn_bids['sz']
        syn_asks['liquidity'] = syn_asks['price'] * syn_asks['sz']

        return syn_bids.reset_index(drop=True), syn_asks.reset_index(drop=True)

    def fetch_all_books(
        base_asset: str
    ) -> list:
        books = []
        bids_liquidity = []

        symbols_usdc, symbols_usd = build_symbols(base_asset, "USDC")
        symbols_usdt, _ = build_symbols(base_asset, "USDT")

        # Binance
        try:
            try:
                b_bids_usdc, b_asks_usdc = fetch_orderbook(symbols_usdc["binance"], limit=5000)
            except Exception as e:
                b_bids_usdc, b_asks_usdc = [], []
            b_bids_usdt, b_asks_usdt = fetch_orderbook(symbols_usdt["binance"], limit=5000)
            
            binance_liq_usdc = sum(price*qty for price, qty in b_bids_usdc)
            binance_liq_usdt = sum(price*qty for price, qty in b_bids_usdt)

            b_bids = b_bids_usdc + b_bids_usdt
            b_asks = b_asks_usdc + b_asks_usdt

            books.append(("binance", b_bids, b_asks))
            bids_liquidity.append(("binance_usdc", binance_liq_usdc))
            bids_liquidity.append(("binance_usdt", binance_liq_usdt))
        except Exception as e:
            pass
            # print("Binance error:", e)

        # Coinbase
        try:
            c_bids, c_asks = fetch_coinbase_orderbook(symbols_usd["coinbase"], level=2)
            total_liq = sum(price*qty for price, qty in c_bids)

            books.append(("coinbase", c_bids, c_asks))
            bids_liquidity.append(("coinbase", total_liq))
        except Exception as e:
            pass
            # print("Coinbase error:", e)

        # Bybit
        try:
            try:
                by_bids_usdc, by_asks_usdc = fetch_bybit_orderbook(symbols_usdc["bybit"], limit=1000)
            except Exception as e:
                by_bids_usdc, by_asks_usdc = [], []
            by_bids_usdt, by_asks_usdt = fetch_bybit_orderbook(symbols_usdt["bybit"], limit=1000)

            by_liq_usdc = sum(price*qty for price, qty in by_bids_usdc)
            by_liq_usdt = sum(price*qty for price, qty in by_bids_usdt)
            
            by_bids = by_bids_usdc + by_bids_usdt
            by_asks = by_asks_usdc + by_asks_usdt

            books.append(("bybit", by_bids, by_asks))
            bids_liquidity.append(("bybit_usdc", by_liq_usdc))
            bids_liquidity.append(("bybit_usdt", by_liq_usdt))
        except Exception as e:
            pass
            # print("Bybit error:", e)

        # OKX
        try:
            try:
                o_bids_usdc, o_asks_usdc = fetch_okx_orderbook(symbols_usdc["okx"], depth=400)
            except Exception as e:
                o_bids_usdc, o_asks_usdc = [], []
            o_bids_usdt, o_asks_usdt = fetch_okx_orderbook(symbols_usdt["okx"], depth=400)

            o_liq_usdc = sum(price*qty for price, qty in o_bids_usdc)
            o_liq_usdt = sum(price*qty for price, qty in o_bids_usdt)
            
            o_bids = o_bids_usdc + o_bids_usdt
            o_asks = o_asks_usdc + o_asks_usdt

            books.append(("okx", o_bids, o_asks))
            bids_liquidity.append(("okx_usdc", o_liq_usdc))
            bids_liquidity.append(("okx_usdt", o_liq_usdt))
        except Exception as e:
            pass
            # print("OKX error:", e)

        # Kraken
        try:
            k_bids, k_asks = fetch_kraken_orderbook(symbols_usd["kraken"], count=500)
            k_liq = sum(price*qty for price, qty in k_bids)

            books.append(("kraken", k_bids, k_asks))
            bids_liquidity.append(("kraken", k_liq))
        except Exception as e:
            pass
            # print("Kraken error:", e)

        # Bitget
        try:
            try:
                bg_bids_usdc, bg_asks_usdc = fetch_bitget_orderbook(symbols_usdc["bitget"], depth_type="step0", limit=200)
            except Exception as e:
                bg_bids_usdc, bg_asks_usdc = [], []
            bg_bids_usdt, bg_asks_usdt = fetch_bitget_orderbook(symbols_usdt["bitget"], depth_type="step0", limit=200)

            bg_liq_usdc = sum(price*qty for price, qty in bg_bids_usdc)
            bg_liq_usdt = sum(price*qty for price, qty in bg_bids_usdt)
            
            bg_bids = bg_bids_usdc + bg_bids_usdt
            bg_asks = bg_asks_usdc + bg_asks_usdt

            books.append(("bitget", bg_bids, bg_asks))
            bids_liquidity.append(("bitget_usdc", bg_liq_usdc))
            bids_liquidity.append(("bitget_usdt", bg_liq_usdt))
        except Exception as e:
            pass
            # print("Bitget error:", e)

        # Gate.io
        try:
            try:
                g_bids_usdc, g_asks_usdc = fetch_gate_orderbook(symbols_usdc["gate"], limit=100, interval="0")
            except Exception as e:
                g_bids_usdc, g_asks_usdc = [], []
            g_bids_usdt, g_asks_usdt = fetch_gate_orderbook(symbols_usdt["gate"], limit=100, interval="0")

            g_liq_usdc = sum(price*qty for price, qty in g_bids_usdc)
            g_liq_usdt = sum(price*qty for price, qty in g_bids_usdt)
            
            g_bids = g_bids_usdc + g_bids_usdt
            g_asks = g_asks_usdc + g_asks_usdt
            
            books.append(("gate", g_bids, g_asks))
            bids_liquidity.append(("gate_usdc", g_liq_usdc))
            bids_liquidity.append(("gate_usdt", g_liq_usdt))
        except Exception as e:
            pass
            # print("Gate error:", e)

        # KuCoin
        try:
            try:
                kuc_bids_usdc, kuc_asks_usdc = fetch_kucoin_orderbook(symbols_usdc["kucoin"], trade_type="SPOT", limit="100")
            except Exception as e:
                kuc_bids_usdc, kuc_asks_usdc = [], []
            kuc_bids_usdt, kuc_asks_usdt = fetch_kucoin_orderbook(symbols_usdt["kucoin"], trade_type="SPOT", limit="100")

            kuc_liq_usdc = sum(price*qty for price, qty in kuc_bids_usdc)
            kuc_liq_usdt = sum(price*qty for price, qty in kuc_bids_usdt)

            kuc_bids = kuc_bids_usdc + kuc_bids_usdt
            kuc_asks = kuc_asks_usdc + kuc_asks_usdt

            books.append(("kucoin", kuc_bids, kuc_asks))
            bids_liquidity.append(("kucoin_usdc", kuc_liq_usdc))
            bids_liquidity.append(("kucoin_usdt", kuc_liq_usdt))
        except Exception as e:
            pass
            # print("KuCoin error:", e)

        # Huobi
        try:
            try:
                h_bids_usdc, h_asks_usdc = fetch_huobi_orderbook(symbols_usdc["huobi"], depth=150, level="step0")
            except Exception as e:
                h_bids_usdc, h_asks_usdc = [], []
            h_bids_usdt, h_asks_usdt = fetch_huobi_orderbook(symbols_usdt["huobi"], depth=150, level="step0")

            h_liq_usdc = sum(price*qty for price, qty in h_bids_usdc)
            h_liq_usdt = sum(price*qty for price, qty in h_bids_usdt)

            h_bids = h_bids_usdc + h_bids_usdt
            h_asks = h_asks_usdc + h_asks_usdt

            books.append(("huobi", h_bids, h_asks))
            bids_liquidity.append(("huobi_usdc", h_liq_usdc))
            bids_liquidity.append(("huobi_usdt", h_liq_usdt))
        except Exception as e:
            pass
            # print("Huobi error:", e)

        # Crypto.com
        try:
            try:
                cc_bids_usdc, cc_asks_usdc = fetch_cryptocom_orderbook(symbols_usdc["cryptocom"], depth=50)
            except Exception as e:
                cc_bids_usdc, cc_asks_usdc = [], []
            cc_bids_usdt, cc_asks_usdt = fetch_cryptocom_orderbook(symbols_usdt["cryptocom"], depth=50)

            cc_liq_usdc = sum(price*qty for price, qty in cc_bids_usdc)
            cc_liq_usdt = sum(price*qty for price, qty in cc_bids_usdt)

            cc_bids = cc_bids_usdc + cc_bids_usdt
            cc_asks = cc_asks_usdc + cc_asks_usdt

            books.append(("cryptocom", cc_bids, cc_asks))
            bids_liquidity.append(("cryptocom_usdc", cc_liq_usdc))
            bids_liquidity.append(("cryptocom_usdt", cc_liq_usdt))
        except Exception as e:
            pass
            # print("Crypto.com error:", e)

        # Bitfinex
        try:
            bf_bids, bf_asks = fetch_bitfinex_orderbook(symbols_usd["bitfinex"], precision="P0", length=100)
            bf_liq_usdc = sum(price*qty for price, qty in bf_bids)

            books.append(("bitfinex", bf_bids, bf_asks))
            bids_liquidity.append(("bitfinex", bf_liq_usdc))
        except Exception as e:
            pass
            # print("Bitfinex error:", e)

        return books

    books = fetch_all_books(base_asset)
    syn_bids, syn_asks = aggregate_books(books)

    return syn_bids, syn_asks

all_sell_orderbooks = {}
for coll_token in collateral_tokens:

    # Slippage and Orderbook Data
    if coll_token == "CBBTC":
        # pool_address = "0xfBB6Eed8e7aa03B138556eeDaF5D271A5E1e43ef"
        # multicall_address = "0xca11bde05977b3631167028862be2a173976ca11"
        # all_sell_orderbooks[coll_token], _ = get_univ3_orderbook(pool_address, multicall_address)
        continue

    elif "HYPE" in coll_token:
        all_sell_orderbooks[coll_token], _ = get_hyper_orderbook(coll_token)

    elif "SOL" in coll_token:
        all_sell_orderbooks[coll_token], _ = get_cex_orderbook("SOL")

    elif "ETH" in coll_token:
        all_sell_orderbooks[coll_token], _ = get_cex_orderbook("ETH")

    elif "BTC " in coll_token:
        all_sell_orderbooks[coll_token], _ = get_cex_orderbook("BTC")

    else:
        all_sell_orderbooks[coll_token], _ = get_cex_orderbook(coll_token)


# Save as parquet one for each collateral

for coll_token, orderbook in all_sell_orderbooks.items():
    if coll_token == "ETH" or "SOL" in coll_token:
        filename = f"{coll_token.lower()}_sell_orderbook.parquet"
        orderbook.to_parquet(filename)
